# Lab 07 — LangGraph Node Graph

## Objective

Build a typed state graph with successful and failed routes.

## Introduction and concept

build_graph uses StateGraph and a conditional edge. reject clears candidate questions and records a terminal status.

## Real-world use case

An exam builder should reject question sets that do not meet the requested count.

## Technology stack and dependencies

Python TypedDict and LangGraph. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

START → generate → validate → finish or reject → END


## Step 1 — Implement the components

build_graph uses StateGraph and a conditional edge. reject clears candidate questions and records a terminal status.

In [1]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

class ExamState(TypedDict, total=False):
    count: int
    questions: list[str]
    valid: bool
    status: str
    trace: list[str]

def generate(state: ExamState):
    bank = ["What is a primary key?", "What is an inner join?"]
    return {"questions": bank[:max(state["count"], 0)], "trace": ["generate"]}

def validate(state: ExamState):
    valid = state["count"] > 0 and len(state["questions"]) == state["count"]
    return {"valid": valid, "trace": state["trace"] + ["validate"]}

def finish(state: ExamState):
    return {"status": "ready", "trace": state["trace"] + ["finish"]}

def reject(state: ExamState):
    return {"status": "rejected", "questions": [], "trace": state["trace"] + ["reject"]}

def build_graph():
    graph = StateGraph(ExamState)
    for node in [generate, validate, finish, reject]:
        graph.add_node(node.__name__, node)
    graph.add_edge(START, "generate")
    graph.add_edge("generate", "validate")
    graph.add_conditional_edges(
        "validate", lambda state: "ready" if state["valid"] else "invalid",
        {"ready": "finish", "invalid": "reject"},
    )
    graph.add_edge("finish", END)
    graph.add_edge("reject", END)
    return graph.compile()

## Step 2 — Run sample inputs and inspect actual output

Request two, three and zero questions. Verify the ready and rejected routes and inspect their traces.

In [2]:
def main():
    graph = build_graph()
    for count, status in [(2, "ready"), (3, "rejected"), (0, "rejected")]:
        result = graph.invoke({"count": count})
        print(count, result["status"], " → ".join(result["trace"]))
        assert result["status"] == status
        assert bool(result["questions"]) == (status == "ready")
    print("PASS: success and failure routes")

main()

2 ready generate → validate → finish
3 rejected generate → validate → reject
0 rejected generate → validate → reject
PASS: success and failure routes


## Testing and validation

Request two, three and zero questions. Verify the ready and rejected routes and inspect their traces. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

Generation uses a fixed bank and validation only checks count; full ExamPilot validates richer constraints.

## How this relates to ExamPilot

Implemented analogue. ExamPilot coordinates exam preparation, delivery and evaluation in backend/app/graph/exam_graph.py.

## Conclusion

The lab builds a typed state graph with successful and failed routes. Use the trace and assertions to explain which decisions the workflow makes.